In [7]:
import pandas as pd
from pathlib import Path
from sqlalchemy import create_engine
import os

In [8]:
DB_URL = (
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

engine = create_engine(DB_URL)

print("Engine listo")

Engine listo


In [6]:
from sqlalchemy import text

query = """
ALTER TABLE lexdata.hechos_vif
ADD COLUMN sexo_de_la_victima TEXT,
ADD COLUMN escolaridad TEXT;
"""

with engine.connect() as con:
    con.execute(text(query))
    con.commit()

print("✅ Tabla modificada")

✅ Tabla modificada


In [12]:
# 📂 ruta completa
base = Path(r"C:\Users\Acer\OneDrive\Escritorio\CARPETAS\septimo semestre\data thinking\2segunda entrega\LexData\notebooks\03_data_judicial")

# 📦 archivos → tablas
archivos = [
    ("lexdata_vif_inmlcf.csv", "hechos_vif"),
    ("lexdata_vif_policia.csv", "hechos_vif"),
    ("lexdata_inasistencia_alimentaria.csv", "inasistencia_alimentaria"),
    ("lexdata_icbf_medidas.csv", "medidas_icbf"),
    ("lexdata_csj_alimentos.csv", "procesos_alimentos"),
    ("lexdata_ivf_resumen_municipios_v9.csv", "ivf_municipios"),
]

# 🧱 esquema esperado en PostgreSQL
columnas_validas = {
    "hechos_vif": [
        "municipio",
        "departamento",
        "anio",
        "cantidad",
        "tipo_ciclo",
        "fuente"
    ],
    "inasistencia_alimentaria": [
        "municipio",
        "departamento",
        "anio",
        "cantidad",
        "tipo_ciclo",
        "fuente"
    ],
    "medidas_icbf": [
        "municipio",
        "departamento",
        "anio",
        "cantidad",
        "tipo_ciclo",
        "fuente"
    ],
    "procesos_alimentos": [
        "municipio",
        "departamento",
        "anio",
        "cantidad",
        "tipo_ciclo",
        "fuente"
    ],
    "ivf_municipios": [
        "municipio",
        "anio",
        "vif_total",
        "alimentos_familia_total",
        "medidas_proteccion_total",
        "inasistencia_total",
        "ivf_score_bruto",
        "ivf_score_ponderado",
        "ivf_tasa_100k",
        "nivel_riesgo",
        "poblacion",
        "alerta"
    ]
}

# 🚀 ETL
for archivo, tabla in archivos:

    ruta = base / archivo
    print(f"\n📂 Cargando {archivo} → {tabla}")

    df = pd.read_csv(ruta, encoding="utf-8", low_memory=False)

    df.columns = df.columns.str.strip().str.lower()

    df = df.rename(columns={
        "año": "anio",
        "anio ": "anio",
        "ano": "anio"
    })

    cols = columnas_validas[tabla]
    cols_ok = [c for c in cols if c in df.columns]

    df = df[cols_ok]

    # 🔥 FIX ESPECIAL IVF (CRÍTICO)
    if tabla == "ivf_municipios":
        df = df.dropna(subset=["municipio"])

    df.to_sql(
        tabla,
        engine,
        schema="lexdata",
        if_exists="append",
        index=False
    )

    print(f"✅ {tabla} cargada correctamente")


📂 Cargando lexdata_vif_inmlcf.csv → hechos_vif
✅ hechos_vif cargada correctamente

📂 Cargando lexdata_vif_policia.csv → hechos_vif
✅ hechos_vif cargada correctamente

📂 Cargando lexdata_inasistencia_alimentaria.csv → inasistencia_alimentaria
✅ inasistencia_alimentaria cargada correctamente

📂 Cargando lexdata_icbf_medidas.csv → medidas_icbf
✅ medidas_icbf cargada correctamente

📂 Cargando lexdata_csj_alimentos.csv → procesos_alimentos
✅ procesos_alimentos cargada correctamente

📂 Cargando lexdata_ivf_resumen_municipios_v9.csv → ivf_municipios
✅ ivf_municipios cargada correctamente
